# Week 2 — Exploratory Data Analysis

Cheater vs. legitimate-player behaviour in Counter-Strike view-angle telemetry.
This notebook is a narrative companion to `reports/eda_findings.md`; the heavy
feature build lives in `scripts/02_build_features.py`. We work on a sample here
so it runs in seconds.


In [1]:
import sys, pathlib
import numpy as np, pandas as pd
for cand in [pathlib.Path.cwd()/"src", pathlib.Path.cwd().parent/"src"]:
    if cand.exists():
        sys.path.insert(0, str(cand)); break
from cheatdetect import config as C, data_loading as D, features as F

def load_sample(label, n):
    a = D.load_class(label)
    idx = np.linspace(0, a.shape[0]-1, min(n, a.shape[0])).astype(int)
    return np.asarray(a[idx], dtype=np.float32)

SAMPLE = 1500
Cb, Lb = load_sample(1, SAMPLE), load_sample(0, SAMPLE)
print("cheater sample", Cb.shape, "| legit sample", Lb.shape)

cheater sample (1500, 30, 192, 5) | legit sample (1500, 30, 192, 5)


## 1. Schema recap

Each instance is `30 engagements x 192 ticks x 5 channels`; channels are
`[d_yaw, d_pitch, yaw, pitch, fire]`. Channels 0/1 are angular *velocity*, 2/3
are absolute view angles, 4 is a shot flag (verified in `reports/data_schema.md`).

In [2]:
pd.DataFrame(D.basic_stats(D.load_class(1), sample=2000)).T.round(3)

,min,max,mean,std,p1,p50,p99,frac_zero
d_yaw,-179.698,179.687,-0.005,3.319,-10.690,0.000,10.415,0.319
d_pitch,-98.602,96.938,0.014,0.721,-1.664,0.000,1.659,0.406
yaw,-180.000,180.000,0.861,39.642,-128.963,0.205,131.840,0.001
pitch,-104.879,119.461,-2.541,7.506,-25.379,-1.583,17.703,0.002
fire,0.000,1.000,0.051,0.220,0.000,0.000,1.000,0.949


## 2. Why global aggregates fail

Per-tick aim speed is almost identical between classes — coarse means cannot
separate subtle cheaters from skilled players.

In [3]:
speed = lambda B: np.sqrt(B[...,0]**2 + B[...,1]**2).ravel()
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
for name, B, c in [("cheater", Cb, "#c0392b"), ("legit", Lb, "#2c7fb8")]:
    ax.hist(np.clip(speed(B),0,20), bins=80, density=True, histtype="step", lw=2, color=c, label=name)
ax.set_xlabel("aim speed (deg/tick)"); ax.set_ylabel("density"); ax.legend(); ax.set_title("Per-tick aim speed (overlapping)")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 3. Behavioural features + univariate separability

Reduce each instance to the 39 features, then rank by single-feature AUC
(separating power; 0.50 = useless). The top features are all **shot-centric**.

In [ ]:
fc, fl = F.extract_chunk(Cb), F.extract_chunk(Lb)
def auc(pos, neg):
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    allv = np.concatenate([pos, neg]); order = allv.argsort(kind="mergesort")
    r = np.empty(len(allv)); r[order] = np.arange(1, len(allv)+1)
    a = (r[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg))
    return max(a, 1-a)
tbl = pd.DataFrame([(k, np.nanmean(fc[k]), np.nanmean(fl[k]), auc(fc[k], fl[k])) for k in fc],
                   columns=["feature","cheater_mean","legit_mean","auc"]).sort_values("auc", ascending=False)
tbl.head(12).round(3).reset_index(drop=True)

## 4. The money plot — behaviour around the shot

Cheaters fire with the crosshair already settled: lower aim speed **at** and
**just before** the shot. This is the human target-acquisition signature
(flick → brake → micro-correct → fire) being muted by aim assistance.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11,4))
for ax, key, title in [(axes[0],"speed_at_shot_mean","aim speed AT shot"),
                       (axes[1],"speed_pre_shot_mean","aim speed BEFORE shot")]:
    for name, f, c in [("cheater", fc, "#c0392b"), ("legit", fl, "#2c7fb8")]:
        ax.hist(np.clip(f[key],0,3), bins=40, density=True, histtype="step", lw=2, color=c, label=name)
    ax.set_title(title); ax.set_xlabel("deg/tick"); ax.legend()
plt.show()

## 5. Takeaways

- No single feature separates the classes (best AUC ~0.63) — these are subtle,
  humanised cheaters. The signal is shot-centric and distributional.
- Combined, the features reach ~0.72 ROC-AUC in a leakage-controlled smoke test
  (`scripts/03_check_features.py`); real modelling is Week 4.
- See `reports/eda_findings.md` and `reports/feature_rationale.md` for the full
  write-up and the per-feature reasoning.